In [1]:
import pandas as pd
import numpy as np

from astropathdb import AstroDB

c:\Users\Michael\miniconda3\envs\astroHome\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


In [6]:
db01 = AstroDB(database="wsi01")

# Functions

In [3]:
def format_axis(db, table, phenos, region, size):

    data = db.query(f"SELECT * from {table}").sort_values(["sampleid", "Total_Count"], ascending=[True, False]).reset_index(drop=True)

    data["config_id"] = data[phenos].astype(str).agg("_".join, axis=1)

    data = data[data["config_id"] != "0_0_0_0_0"].reset_index(drop=True)

    data = data.drop(columns=["Other", *phenos])

    data["config"] = data["config_id"]

    data["region"] = region

    data["config_id"] = data["config_id"] + f"_{region}_{size}"

    # data = data[data.groupby("config_id")["Total_Count"].transform('sum') >= min_count].reset_index(drop=True)

    return data


def format_den(df, area, clin=None):

    df_formatted = df.copy()

    df_formatted = df_formatted.merge(area, on=["sampleid", "region"], how="left")

    df_formatted = df_formatted.drop(columns=["region"])

    df_formatted["density"] = df_formatted["Total_Count"] / df_formatted["area"]

    df_formatted = df_formatted.replace([np.inf, -np.inf], 0)

    df_formatted = df_formatted.pivot_table(index="config_id", columns="sampleid", values="density").reset_index().fillna(0)

    df_formatted.columns = ["Features", *[f"sampleid_{x}" for x in df_formatted.columns[1:]]]

    if clin is not None:

        df_formatted = pd.concat([clin, df_formatted]).reset_index(drop=True)

    return df_formatted

# WSI01

## 25 microns

In [4]:
# need to rearrange order 
phenos = ["CD8", "FoxP3", "Tumor", "CD68", "CD8FoxP3"]

size = "25"

In [7]:
region = "tumor"

table = "test.dbo.wsi01_TumorConfigs_25"

pre_tum = format_axis(db01, table, phenos, region, size)

In [8]:
region = "background"

table = "test.dbo.wsi01_BackgroundConfigs_25"

pre_back = format_axis(db01, table, phenos, region, size)

In [9]:
pre_total = pd.concat([pre_tum, pre_back], ignore_index=True).groupby(["sampleid", "config"])["Total_Count"].sum().reset_index()

pre_total["config_id"] = pre_total["config"] + f"_total_{size}"

pre_total = pre_total.drop(columns=["config"])

pre_tum = pre_tum.drop(columns=["config"])

pre_back = pre_back.drop(columns=["config"])

In [10]:
pre_combined = pd.concat([pre_tum, pre_back, pre_total], ignore_index=True).fillna("total")

In [95]:
pre_combined

,sampleid,Total_Count,config_id,region
0,1,501,8_0_0_0_0_tumor_25,tumor
1,1,461,7_0_0_0_0_tumor_25,tumor
2,1,421,9_0_0_0_0_tumor_25,tumor
3,1,402,6_0_0_0_0_tumor_25,tumor
4,1,388,10_0_0_0_0_tumor_25,tumor
...,...,...,...,...
1964858,79,4,9_2_1_0_0_total_25,total
1964859,79,2,9_3_0_0_0_total_25,total
1964860,79,1,9_3_0_0_1_total_25,total
1964861,79,1,9_3_1_0_0_total_25,total


In [10]:
# Get areas
nur = 8000.0000 / (1.004 * 1.344)

area_sql = f"""
    SELECT
        rc.sampleid,
        COUNT(*) / {nur} area
    FROM dbo.randomcell rc
    where (tdist <= 0)
    and rc.sampleid in ({','.join(map(str, pre_combined["sampleid"].unique()))})
    GROUP BY rc.sampleid
    ORDER BY rc.sampleid
    """
area_tum = db01.query(area_sql)

area_tum["region"] = "tumor"

area_sql = f"""
    SELECT
        rc.sampleid,
        COUNT(*) / {nur} area
    FROM dbo.randomcell rc
    where (tdist > 0)
    and rc.sampleid in ({','.join(map(str, pre_combined["sampleid"].unique()))})
    GROUP BY rc.sampleid
    ORDER BY rc.sampleid
    """
area_back = db01.query(area_sql)

area_back["region"] = "background"

area = pd.concat([area_tum, area_back], ignore_index=True)

area_total = area.groupby("sampleid")["area"].sum().reset_index()

area_total["region"] = "total"

area = pd.concat([area, area_total], ignore_index=True)

In [ ]:
combined_25 = format_den(pre_combined, area)

In [ ]:
combined_25

,Features,sampleid_1,sampleid_2,sampleid_3,sampleid_4,sampleid_5,sampleid_6,sampleid_7,sampleid_8,sampleid_9,...,sampleid_68,sampleid_69,sampleid_71,sampleid_72,sampleid_73,sampleid_74,sampleid_76,sampleid_77,sampleid_78,sampleid_79
0,0_0_0_0_10_background_25,0.217782,0.000000,0.021297,0.150397,0.072827,0.021092,0.140367,0.083360,0.000000,...,0.000000,1.455545,1.462865,0.0,1.107748,0.785612,0.072392,1.178297,0.143866,0.894933
1,0_0_0_0_10_total_25,1.279820,0.034091,0.172023,0.721602,0.175333,0.108831,0.128261,0.102458,0.765549,...,0.063317,5.414078,4.376955,0.0,1.418681,8.556477,0.160133,9.824647,2.143112,9.052938
2,0_0_0_0_10_tumor_25,0.000000,0.050380,0.018679,0.036564,0.000000,0.005044,0.000000,0.018304,0.070041,...,0.016017,0.192322,0.455631,0.0,0.000000,0.421725,0.000000,0.959486,0.050961,0.443623
3,0_0_0_0_11_background_25,0.140412,0.000000,0.015643,0.150397,0.182069,0.013759,0.140367,0.066688,0.000000,...,0.000000,1.118123,1.219054,0.0,0.830811,0.629634,0.000000,1.466325,0.090144,0.482399
4,0_0_0_0_11_total_25,0.471513,0.000000,0.129017,0.736635,0.175333,0.042873,0.064130,0.079690,0.291638,...,0.050653,3.544267,1.783204,0.0,0.354670,6.514591,0.000000,4.742933,1.306203,5.269093
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95430,9_9_4_1_0_tumor_25,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
95431,9_9_5_0_0_background_25,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
95432,9_9_5_0_0_total_25,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
95433,9_9_5_0_1_total_25,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 17.5 microns

In [97]:
# need to rearrange order 
phenos = ["CD8", "FoxP3", "Tumor", "CD68", "CD8FoxP3"]

size = "175"

In [98]:
region = "tumor"

table = "test.dbo.wsi01_TumorConfigs_175"

pre_tum = format_axis(db01, table, phenos, region, size)

In [99]:
region = "background"

table = "test.dbo.wsi01_BackgroundConfigs_175"

pre_back = format_axis(db01, table, phenos, region, size)

In [100]:
pre_total = pd.concat([pre_tum, pre_back], ignore_index=True).groupby(["sampleid", "config"])["Total_Count"].sum().reset_index()

pre_total["config_id"] = pre_total["config"] + f"_total_{size}"

pre_total = pre_total.drop(columns=["config"])

pre_tum = pre_tum.drop(columns=["config"])

pre_back = pre_back.drop(columns=["config"])

In [101]:
pre_combined = pd.concat([pre_tum, pre_back, pre_total], ignore_index=True).fillna("total")

In [102]:
pre_combined

,sampleid,Total_Count,config_id,region
0,1,1325,3_0_0_0_0_tumor_175,tumor
1,1,1246,4_0_0_0_0_tumor_175,tumor
2,1,1060,5_0_0_0_0_tumor_175,tumor
3,1,997,2_0_0_0_0_tumor_175,tumor
4,1,672,6_0_0_0_0_tumor_175,tumor
...,...,...,...,...
577883,79,1,9_0_0_0_3_total_175,total
577884,79,7,9_0_1_0_0_total_175,total
577885,79,1,9_0_1_0_1_total_175,total
577886,79,1,9_1_0_0_0_total_175,total


In [31]:
# Get areas
nur = 8000.0000 / (1.004 * 1.344)

area_sql = f"""
    SELECT
        rc.sampleid,
        COUNT(*) / {nur} 'area'
    FROM dbo.randomcell rc
    where (tdist <= 0)
    and rc.sampleid in ({','.join(map(str, pre_combined["sampleid"].unique()))})
    GROUP BY rc.sampleid
    ORDER BY rc.sampleid
    """
area_tum = db01.query(area_sql)

area_tum["region"] = "tumor"

area_sql = f"""
    SELECT
        rc.sampleid,
        COUNT(*) / {nur} 'area'
    FROM dbo.randomcell rc
    where (tdist > 0)
    and rc.sampleid in ({','.join(map(str, pre_combined["sampleid"].unique()))})
    GROUP BY rc.sampleid
    ORDER BY rc.sampleid
    """
area_back = db01.query(area_sql)

area_back["region"] = "background"

area = pd.concat([area_tum, area_back], ignore_index=True)

area_total = area.groupby("sampleid")["area"].sum().reset_index()

area_total["region"] = "total"

area = pd.concat([area, area_total], ignore_index=True)

In [ ]:
combined_175 = format_den(pre_combined, area)

In [ ]:
combined_175

,Features,sampleid_1,sampleid_2,sampleid_3,sampleid_4,sampleid_5,sampleid_6,sampleid_7,sampleid_8,sampleid_9,...,sampleid_68,sampleid_69,sampleid_71,sampleid_72,sampleid_73,sampleid_74,sampleid_76,sampleid_77,sampleid_78,sampleid_79
0,0_0_0_0_10_background_175,0.240706,0.000000,0.021757,0.241710,0.072827,0.011430,0.0,0.000000,0.000000,...,0.000000,0.416286,1.083604,0.000000,0.0,0.547353,0.0,0.523688,0.055543,0.289439
1,0_0_0_0_10_total_175,0.202077,0.034091,0.114682,0.435968,0.035067,0.019787,0.0,0.011384,0.255183,...,0.151960,0.893044,0.648438,1.228994,0.0,4.472704,0.0,0.677562,0.683084,0.919439
2,0_0_0_0_10_tumor_175,0.000000,0.025190,0.023773,0.041135,0.000000,0.000000,0.0,0.013728,0.064204,...,0.076880,0.085931,0.000000,1.324250,0.0,0.905927,0.0,0.000000,0.045885,0.125967
3,0_0_0_0_11_background_175,0.000000,0.000000,0.022160,0.112798,0.072827,0.012700,0.0,0.000000,0.000000,...,0.000000,0.630736,0.000000,0.000000,0.0,0.815663,0.0,0.523688,0.035208,0.137234
4,0_0_0_0_11_total_175,0.000000,0.000000,0.050173,0.240534,0.035067,0.013192,0.0,0.011384,0.145819,...,0.050653,0.530245,0.000000,0.000000,0.0,4.278239,0.0,0.338781,0.380650,0.265223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19499,9_8_0_0_1_total_175,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
19500,9_8_1_0_0_background_175,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
19501,9_8_1_0_0_total_175,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
19502,9_9_2_0_1_background_175,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000


## 10 microns

In [104]:
# need to rearrange order 
phenos = ["CD8", "FoxP3", "Tumor", "CD68", "CD8FoxP3"]

size = "10"

In [105]:
region = "tumor"

table = "test.dbo.wsi01_TumorConfigs_10"

pre_tum = format_axis(db01, table, phenos, region, size)

In [106]:
region = "background"

table = "test.dbo.wsi01_BackgroundConfigs_10"

pre_back = format_axis(db01, table, phenos, region, size)

In [107]:
pre_total = pd.concat([pre_tum, pre_back], ignore_index=True).groupby(["sampleid", "config"])["Total_Count"].sum().reset_index()

pre_total["config_id"] = pre_total["config"] + f"_total_{size}"

pre_total = pre_total.drop(columns=["config"])

pre_tum = pre_tum.drop(columns=["config"])

pre_back = pre_back.drop(columns=["config"])

In [108]:
pre_combined = pd.concat([pre_tum, pre_back, pre_total], ignore_index=True).fillna("total")

In [109]:
pre_combined

,sampleid,Total_Count,config_id,region
0,1,3735,1_0_0_0_0_tumor_10,tumor
1,1,2202,2_0_0_0_0_tumor_10,tumor
2,1,880,3_0_0_0_0_tumor_10,tumor
3,1,495,1_0_0_0_0_tumor_10,tumor
4,1,308,0_0_0_0_1_tumor_10,tumor
...,...,...,...,...
78226,79,2,5_0_0_0_1_total_10,total
78227,79,34,6_0_0_0_0_total_10,total
78228,79,1,6_0_0_0_1_total_10,total
78229,79,1,7_0_0_0_0_total_10,total


In [42]:
# Get areas
nur = 8000.0000 / (1.004 * 1.344)

area_sql = f"""
    SELECT
        rc.sampleid,
        COUNT(*) / {nur} 'area'
    FROM dbo.randomcell rc
    where (tdist <= 0)
    and rc.sampleid in ({','.join(map(str, pre_combined["sampleid"].unique()))})
    GROUP BY rc.sampleid
    ORDER BY rc.sampleid
    """
area_tum = db01.query(area_sql)

area_tum["region"] = "tumor"

area_sql = f"""
    SELECT
        rc.sampleid,
        COUNT(*) / {nur} 'area'
    FROM dbo.randomcell rc
    where (tdist > 0)
    and rc.sampleid in ({','.join(map(str, pre_combined["sampleid"].unique()))})
    GROUP BY rc.sampleid
    ORDER BY rc.sampleid
    """
area_back = db01.query(area_sql)

area_back["region"] = "background"

area = pd.concat([area_tum, area_back], ignore_index=True)

area_total = area.groupby("sampleid")["area"].sum().reset_index()

area_total["region"] = "total"

area = pd.concat([area, area_total], ignore_index=True)

In [ ]:
combined_10 = format_den(pre_combined, area)

In [ ]:
combined_10

,Features,sampleid_1,sampleid_2,sampleid_3,sampleid_4,sampleid_5,sampleid_6,sampleid_7,sampleid_8,sampleid_9,...,sampleid_68,sampleid_69,sampleid_71,sampleid_72,sampleid_73,sampleid_74,sampleid_76,sampleid_77,sampleid_78,sampleid_79
0,0_0_0_0_10_background_10,0.0,0.0,0.012087,0.084599,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0
1,0_0_0_0_10_total_10,0.0,0.0,0.003584,0.015033,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.027908,0.0,0.000000,0.0,0.000000,0.0,0.0,0.010429,0.0
2,0_0_0_0_10_tumor_10,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.032736,0.0,0.000000,0.0,0.000000,0.0,0.0,0.006849,0.0
3,0_0_0_0_11_background_10,0.0,0.0,0.012087,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0
4,0_0_0_0_11_total_10,0.0,0.0,0.003584,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.002607,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2149,9_0_0_0_0_background_10,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0
2150,9_0_0_0_0_total_10,0.0,0.0,0.078844,0.000000,0.070133,0.257237,0.064130,0.079690,0.072909,...,0.151960,0.000000,0.0,3.686982,0.0,0.194465,0.0,0.0,0.816051,0.0
2151,9_0_0_0_0_tumor_10,0.0,0.0,0.112072,0.000000,0.135262,0.196732,0.118077,0.096094,0.073376,...,0.153759,0.000000,0.0,3.972749,0.0,0.156194,0.0,0.0,0.535897,0.0
2152,9_0_0_0_1_total_10,0.0,0.0,0.003584,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.010429,0.0


In [ ]:
combined = pd.concat([combined_10, combined_175, combined_25], ignore_index=True)